# Retrieval experiments

Everything tried while building the retrieval step, scored on the tuning set (hit@3/5/10, MRR).
The baseline is a RAG built from scratch with numpy; the final pipeline lives in `src/football_rag`.

In [1]:
import json
import re

import numpy as np
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from langchain_openai import OpenAIEmbeddings

from football_rag import config
from football_rag.embeddings import get_embeddings
from football_rag.ingest import load_docs, split_docs, add_breadcrumbs

In [4]:
CLIENT = OpenAI(base_url=config.LMSTUDIO_URL, api_key=config.LMSTUDIO_API_KEY)
QWEN_EMBEDDING_MODEL = 'text-embedding-qwen3-embedding-0.6b'


def load_eval_set(name):
    """Load one eval set from data/eval by file name."""
    path = config.EVAL_DIR / f'{name}.json'
    with open(path, encoding='utf-8') as f:
        eval_set = json.load(f)
    return eval_set


tuning_set = load_eval_set('hand_written') + load_eval_set('tuning')
print(len(tuning_set))

48


## 1. Chunking

In [5]:
def split_sections(text):
    """Split a Wikipedia article into (section name, body) pairs on its == headings."""
    current_name = 'Introduction'
    current_lines = []
    sections = []
    for line in text.split('\n'):
        stripped = line.strip()

        if stripped.startswith('==') and stripped.endswith('=='):
            sections.append((current_name, '\n'.join(current_lines).strip()))
            current_name = stripped.strip('= ')
            current_lines = []
        else:
            current_lines.append(line)

    sections.append((current_name, '\n'.join(current_lines).strip()))
    return sections


def fixed_chunk_text(text, chunk_size=200, overlap=40):
    """Split text into chunks of chunk_size words, with overlap words shared between neighbours."""
    words = text.split()
    start = 0
    chunks = []

    while start < len(words):
        chunk = words[start: start + chunk_size]
        chunks.append(' '.join(chunk))

        if start + chunk_size >= len(words):
            break

        start += chunk_size - overlap

    return chunks


def structure_based_chunking(text, title, chunk_size, overlap):
    """Chunk each section separately and prefix every chunk with 'title > section'."""
    sections = split_sections(text)
    all_chunks = []

    for name, body in sections:
        if not body or name in config.SKIP_SECTIONS:
            continue

        chunks = fixed_chunk_text(body, chunk_size, overlap)
        prefixed = [f'{title} > {name}: ' + c for c in chunks]
        all_chunks.extend(prefixed)

    return all_chunks

## RAG from scratch
Numpy cosine search, with switches for query rewriting, hybrid BM25 + RRF and cross-encoder reranking.

In [ ]:
def cosine(a, b):
    """Cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norms = np.linalg.norm(a) * np.linalg.norm(b)
    similarity = dot / norms
    return similarity


def tokenize(text):
    """Lowercase word tokens for BM25."""
    tokens = re.findall(r'\w+', text.lower())
    return tokens


class RAG():
    """RAG built from scratch on numpy, with optimal rewrite, hybrid search, and reranking."""

    def __init__(self, chunks, k=3, use_rewrite=False, use_hybrid=False, reranker_model=None, weight_cos=0.5, weight_bm25=0.5):
        self.chunks = chunks
        self.k = k
        self.use_rewrite = use_rewrite
        self.use_hybrid = use_hybrid
        self.weight_cos = weight_cos
        self.weight_bm25 = weight_bm25

        self.reranker = None
        if reranker_model is not None:
            self.reranker = CrossEncoder(reranker_model)

        tokenized = [tokenize(chunk) for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized)

        prefixed = ['search_document: ' + chunk for chunk in chunks]
        response = CLIENT.embeddings.create(model=config.EMBEDDING_MODEL, input=prefixed)
        self.doc_array = np.array([item.embedding for item in response.data])

    def cosine_ranker(self, question):
        """Rank all chunks by cosine similarity to the question."""
        query = 'search_query: ' + question
        response = CLIENT.embeddings.create(model=config.EMBEDDING_MODEL, input=query)
        q_array = np.array(response.data[0].embedding)

        scores = [cosine(q_array, row) for row in self.doc_array]
        pairs = list(zip(scores, self.chunks))
        ranked = sorted(pairs, key=lambda pair: pair[0], reverse=True)
        return ranked

    def bm25_ranker(self, question):
        """Rank all chunks by BM25 keyword score."""
        scores = self.bm25.get_scores(tokenize(question))
        pairs = list(zip(scores, self.chunks))
        ranked = sorted(pairs, key=lambda pair:pair[0], reverse=True)
        return ranked